## Train on CelebA

In [ ]:
from train import Trainer
from config import config
from model import UNet
from modules import NoiseInjectionPipeline
from denoising_diffusion import NoisePredictionPipeline, DDPM

# Forward diffusion
noise_injection_pipeline = NoiseInjectionPipeline(
    config=config,
    random=True
)

# Noise prediction backbone
unet = UNet(
    in_channels = config["in_channels"], # because not it will operate in latent space and not in RGB Space
    out_channels = config["out_channels"],
    time_embd_dim = config["time_embed"],
    base_channels = 64
)

# Noise prediction pipeline
noise_prediction_pipeline = NoisePredictionPipeline(
    config=config,
    noise_prediction_model=unet
)

# Complete DDPM
ddpm = DDPM(
    noise_injection_pipieline=noise_injection_pipeline,
    noise_prediction_pipeline=noise_prediction_pipeline
)

trainer = Trainer(
    ddpm=ddpm,
    config=config,

    dataset_path="../data/celeba_hq_256",

    batch_size = 8,
    image_size = 256,

    epochs = 600,
    learning_rate = 1e-3,

    validation_split = 0.05,

    checkpoint_dir="./checkpoint/celeba_256",
    checkpoint_every = 10,

    project_name="DiffusionEngine",
    run_name="DDPM-CelebA-HQ-256x256",

    num_workers=4,
    use_amp=True
)
trainer.fit()

C:\Users\Himanshu Singh\AppData\Roaming\Python\Python310\site-packages\huggingface_hub\utils\_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\Himanshu Singh\_netrc.
wandb: Currently logged in as: himanshusr451tehs (himanshusr451tehs-personaluse-com) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


DDPM TRAINER INITIALIZED
Device:          cuda
Mixed precision: True
Latent Denoise : Enabled
Dataset size:    30000
Training images: 28500
Validation:      1500
Batch size:      8
Image size:      256x256
Training steps:  3562 per epoch
W&B run:         https://wandb.ai/himanshusr451tehs-personaluse-com/DiffusionEngine/runs/jhd1ey1z


Epoch 1/600:  31%|███       | 1112/3562 [09:46<21:05,  1.94it/s, loss=0.217110, lr=1.00e-03]

##

In [ ]:
import torch
from sample import sample
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
generated_image = sample(ddpm, config, device)
generated_image = (generated_image + 1.0) / 2.0 # Convert [-1, 1] -> [0, 1]
generated_image = generated_image.clamp(0, 1) # for visualization in matplotlib

# Visualize

# Convert BCHW -> HWC
image = generated_image[0].permute(1, 2, 0).cpu().numpy()

plt.figure(figsize=(6, 6))
plt.imshow(image)
plt.axis("off")
plt.title("Generated Image")
plt.show()